DynamoDB is AWS's fully managed NoSQL database — a key-value and document store that delivers single-digit millisecond performance at any scale, with no servers to manage, no schema to define upfront, and automatic replication across multiple AZs. It is one of the most important services on the SAA-C03 exam and the foundation of serverless architectures on AWS.

## Core Concepts

### Tables, Items, Attributes

| Concept | Description | SQL equivalent |
|---|---|---|
| **Table** | Container for data | Table |
| **Item** | A single data record | Row |
| **Attribute** | A field on an item | Column |

DynamoDB is **schemaless** — items in the same table can have different attributes. Only the primary key attributes are required on every item.

### Primary Key

The primary key uniquely identifies each item. Two options:

| Type | Structure | Use case |
|---|---|---|
| **Simple (Partition Key only)** | `PK` | Each item has a unique partition key (e.g. `userId`) |
| **Composite (Partition Key + Sort Key)** | `PK + SK` | Multiple items share a partition key, sorted by SK (e.g. `userId` + `orderDate`) |

### How partition key works
DynamoDB hashes the partition key to determine which physical partition stores the item. Good partition key design = **high cardinality** (many distinct values) = even data distribution = uniform throughput.

Bad partition keys (low cardinality like `status = active/inactive`) create **hot partitions** — one partition gets most traffic and throttles.

### Data types
- **Scalar**: String (`S`), Number (`N`), Binary (`B`), Boolean (`BOOL`), Null (`NULL`)
- **Set**: String Set (`SS`), Number Set (`NS`), Binary Set (`BS`) — no duplicates
- **Document**: List (`L`) — ordered, mixed types; Map (`M`) — nested key-value

## Read/Write Capacity Modes

### Provisioned mode
You specify **Read Capacity Units (RCUs)** and **Write Capacity Units (WCUs)** upfront.

| Unit | What it provides |
|---|---|
| **1 RCU** | 1 strongly consistent read/sec OR 2 eventually consistent reads/sec for items up to 4 KB |
| **1 WCU** | 1 write/sec for items up to 1 KB |

**RCU calculation examples:**
- Read 10 items/sec, each 4 KB, strongly consistent → 10 RCUs
- Read 10 items/sec, each 4 KB, eventually consistent → 5 RCUs
- Read 10 items/sec, each 8 KB, strongly consistent → 20 RCUs (4 KB per RCU → 2 RCUs per item)

**WCU calculation examples:**
- Write 10 items/sec, each 1 KB → 10 WCUs
- Write 10 items/sec, each 2.5 KB → 30 WCUs (round up to 3 KB → 3 WCUs per item)

**Auto Scaling**: Set min/max RCU/WCU and a target utilisation percentage. DynamoDB adjusts capacity automatically based on actual traffic.

### On-demand mode
No capacity planning — DynamoDB scales instantly to handle any traffic level. You pay per request.

| | Provisioned | On-demand |
|---|---|---|
| Cost | Lower (predictable traffic) | Higher per request |
| Planning | Requires capacity estimation | None |
| Scaling | Auto Scaling (minutes) | Instant |
| Use case | Predictable, steady workloads | Spiky, unpredictable, new tables |

> **Exam tip:** On-demand mode cannot throttle — it handles any spike instantly. Provisioned mode can throttle if traffic exceeds configured capacity.

## Read Consistency

DynamoDB replicates data across 3 AZs. This creates a consistency trade-off:

| Model | Description | Cost | Use case |
|---|---|---|---|
| **Eventually consistent** | May return slightly stale data (replication lag) | 0.5 RCU per 4 KB | Default; most reads |
| **Strongly consistent** | Always returns the latest committed data | 1 RCU per 4 KB | When you need the most current data |
| **Transactional** | ACID across multiple items/tables | 2 RCUs per 4 KB | Financial transactions, inventory |

Strongly consistent reads cost **2× more** than eventually consistent. Transactional reads cost **4× more**.

## Secondary Indexes

Secondary indexes let you query on attributes other than the primary key.

### Local Secondary Index (LSI)
- **Same partition key** as the base table, **different sort key**
- Must be created **at table creation time** — cannot be added later
- Up to **5 LSIs** per table
- Shares the base table's RCU/WCU (not separate throughput)
- Supports **strongly consistent reads**
- Max size per partition key value: **10 GB** (table + all LSIs)

### Global Secondary Index (GSI)
- **Different partition key** (and optional sort key) from the base table
- Can be added or deleted **at any time**
- Up to **20 GSIs** per table (default limit)
- Has its **own separate RCU/WCU** (provisioned independently)
- Only supports **eventually consistent reads**
- No size limit per partition

### LSI vs GSI

| | LSI | GSI |
|---|---|---|
| Partition key | Same as table | Different |
| When created | Table creation only | Anytime |
| Consistency | Strongly consistent ✅ | Eventually consistent only |
| Throughput | Shared with table | Separate |
| Size limit | 10 GB per PK | None |

> **Exam tip:** If you need strongly consistent reads on an alternate key, use LSI (and plan at table design time). If you need to query on a completely different key after the table is created, use GSI.

## DynamoDB Operations

### Single-item operations
| Operation | Description |
|---|---|
| `GetItem` | Read a single item by primary key |
| `PutItem` | Create or fully replace an item |
| `UpdateItem` | Modify specific attributes of an item (or create if not exists) |
| `DeleteItem` | Delete an item by primary key |

### Multi-item operations
| Operation | Description |
|---|---|
| `Query` | Retrieve items by partition key + optional sort key condition. **Efficient** — reads only the relevant partition. |
| `Scan` | Read **every item** in the table, then filter. **Expensive** — consumes capacity for entire table. |
| `BatchGetItem` | Up to 100 items across tables in one call |
| `BatchWriteItem` | Up to 25 puts/deletes across tables in one call |

### Transactions
| Operation | Description |
|---|---|
| `TransactWriteItems` | All-or-nothing write across up to 100 items/tables |
| `TransactGetItems` | Consistent read across up to 100 items/tables |

Transactions are ACID — if any operation fails, all are rolled back. Cost: 2× WCU for writes, 2× RCU for reads.

> **Scan is almost always wrong for production.** Design your access patterns first, then model your table (primary key + indexes) to support them with Query.

## DynamoDB Accelerator (DAX)

DAX is an **in-memory cache** for DynamoDB — fully managed, DynamoDB-compatible, delivers **microsecond read latency** (vs single-digit milliseconds for DynamoDB).

```text
App → DAX Cluster → DynamoDB
        │
        ├── Cache HIT  → return in microseconds
        └── Cache MISS → fetch from DynamoDB, cache, return
```

### Key properties
- **Write-through**: writes go to DynamoDB first, then DAX cache is updated
- **API compatible**: use the DAX SDK client — same API as DynamoDB SDK
- **Multi-AZ cluster**: primary + replicas across AZs
- **Lives in your VPC**: DAX nodes have private IPs
- Default **TTL**: 5 minutes for item cache, 1 minute for query cache

### When to use DAX vs ElastiCache

| | DAX | ElastiCache |
|---|---|---|
| Works with | DynamoDB only | Any backend |
| API | DynamoDB-compatible (drop-in) | Redis/Memcached API |
| Best for | Read-heavy DynamoDB (GetItem, Query) | Complex query results, computed aggregations |
| Write caching | Write-through | You control |

> Use DAX when you need faster DynamoDB reads with minimal code change. Use ElastiCache when caching complex aggregations or data from multiple sources.

## DynamoDB Streams

DynamoDB Streams captures a **time-ordered sequence of item-level changes** in a table — creates, updates, and deletes.

### Stream record content (configurable)
| View type | What's captured |
|---|---|
| `KEYS_ONLY` | Only the primary key attributes |
| `NEW_IMAGE` | The entire item after the change |
| `OLD_IMAGE` | The entire item before the change |
| `NEW_AND_OLD_IMAGES` | Both before and after |

### Properties
- Records retained for **24 hours**
- Each shard in the stream corresponds to a DynamoDB partition
- Processed by **Lambda** (event source mapping) or Kinesis Data Streams
- Exactly-once delivery to Lambda

### Use cases
```text
DynamoDB item changes → Stream → Lambda
  ├── Trigger notifications (new order created)
  ├── Replicate to ElasticSearch for full-text search
  ├── Audit log to S3
  ├── Cross-region replication (basis for Global Tables)
  └── Trigger downstream microservices
```

## Global Tables

Global Tables is DynamoDB's **multi-region, multi-active** replication feature. Every region can accept reads AND writes, and changes are automatically replicated to all other regions.

```text
us-east-1 table  ←──── replication ────▶  eu-west-1 table
     ↕                                          ↕
  Reads + Writes                          Reads + Writes
```

### Key properties
- **Requires DynamoDB Streams** to be enabled
- **Requires on-demand mode or provisioned with Auto Scaling**
- Replication latency typically **under 1 second**
- Conflict resolution: **last-writer-wins** (based on timestamp)
- No additional code changes — your app reads/writes to the local table

### Use cases
- Global applications needing local write latency in every region
- Active-active disaster recovery
- Data sovereignty — users' data is served from their region

## TTL — Time to Live

TTL automatically deletes items after a specified expiry time — no RCU/WCU cost for the deletions.

- Enable TTL on a table and specify which attribute holds the expiry timestamp (Unix epoch, in seconds)
- DynamoDB deletes expired items within **48 hours** of the expiry time (not immediate)
- Deletions appear in DynamoDB Streams (useful for cleanup triggers)
- Items are not deleted at exactly the TTL time — they can still be returned by reads until actually deleted; use a filter expression to exclude them if needed

### Use cases
- Session data — expire after 30 minutes of inactivity
- Temporary cart items
- Rate limiting counters — expire after the window
- Log entries — keep only last 90 days

## Backup and Restore

| Feature | Details |
|---|---|
| **On-demand backup** | Manual full backup; stored until deleted; no performance impact; cross-region copy supported |
| **PITR** (Point-in-time recovery) | Continuous backups for up to **35 days**; restore to any second; creates a new table |

PITR must be explicitly enabled per table. It protects against accidental writes or deletes.

## Working with DynamoDB via boto3

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key, Attr
from decimal import Decimal

dynamodb = boto3.resource('dynamodb', region_name='us-east-1')
client   = boto3.client('dynamodb',   region_name='us-east-1')

In [ ]:
# Create an orders table with composite key + GSI + LSI
table = dynamodb.create_table(
    TableName='Orders',
    KeySchema=[
        {'AttributeName': 'customerId', 'KeyType': 'HASH'},   # Partition key
        {'AttributeName': 'orderId',    'KeyType': 'RANGE'},  # Sort key
    ],
    AttributeDefinitions=[
        {'AttributeName': 'customerId', 'AttributeType': 'S'},
        {'AttributeName': 'orderId',    'AttributeType': 'S'},
        {'AttributeName': 'orderDate',  'AttributeType': 'S'},
        {'AttributeName': 'status',     'AttributeType': 'S'},
    ],
    LocalSecondaryIndexes=[
        {
            'IndexName': 'CustomerOrderDateIndex',
            'KeySchema': [
                {'AttributeName': 'customerId', 'KeyType': 'HASH'},
                {'AttributeName': 'orderDate',  'KeyType': 'RANGE'}  # different sort key
            ],
            'Projection': {'ProjectionType': 'ALL'}
        }
    ],
    GlobalSecondaryIndexes=[
        {
            'IndexName': 'StatusIndex',
            'KeySchema': [
                {'AttributeName': 'status',    'KeyType': 'HASH'},
                {'AttributeName': 'orderDate', 'KeyType': 'RANGE'}
            ],
            'Projection': {'ProjectionType': 'ALL'},
            'ProvisionedThroughput': {'ReadCapacityUnits': 5, 'WriteCapacityUnits': 5}
        }
    ],
    BillingMode='PAY_PER_REQUEST',   # on-demand
    StreamSpecification={
        'StreamEnabled': True,
        'StreamViewType': 'NEW_AND_OLD_IMAGES'
    }
)
print(f"Table: {table.table_name}, Status: {table.table_status}")

In [ ]:
table = dynamodb.Table('Orders')

# PutItem — create an order
table.put_item(Item={
    'customerId': 'cust-001',
    'orderId':    'order-001',
    'orderDate':  '2026-04-18',
    'status':     'PENDING',
    'amount':     Decimal('59.99'),
    'items':      [{'sku': 'ABC', 'qty': 2}]
})

# UpdateItem — change status without overwriting the whole item
table.update_item(
    Key={'customerId': 'cust-001', 'orderId': 'order-001'},
    UpdateExpression='SET #s = :new_status, updatedAt = :now',
    ExpressionAttributeNames={'#s': 'status'},  # 'status' is a reserved word
    ExpressionAttributeValues={':new_status': 'SHIPPED', ':now': '2026-04-19'}
)

# GetItem — fetch by primary key
response = table.get_item(
    Key={'customerId': 'cust-001', 'orderId': 'order-001'},
    ConsistentRead=True  # strongly consistent
)
print(response['Item'])

In [ ]:
# Query — efficient: reads only items for customerId='cust-001'
response = table.query(
    KeyConditionExpression=Key('customerId').eq('cust-001') &
                           Key('orderId').begins_with('order-'),
    FilterExpression=Attr('status').eq('SHIPPED'),  # applied after read — still charges for all matching PK items
    ScanIndexForward=False  # descending sort key order
)
print(f"Found {response['Count']} orders")

# Query a GSI — find all PENDING orders on a specific date
response = table.query(
    IndexName='StatusIndex',
    KeyConditionExpression=Key('status').eq('PENDING') &
                           Key('orderDate').eq('2026-04-18')
)
print(f"Pending orders today: {response['Count']}")

In [ ]:
# TransactWriteItems — deduct inventory AND create order atomically
client.transact_write_items(
    TransactItems=[
        {
            'Update': {
                'TableName': 'Inventory',
                'Key': {'sku': {'S': 'ABC'}},
                'UpdateExpression': 'SET quantity = quantity - :qty',
                'ConditionExpression': 'quantity >= :qty',  # fail if not enough stock
                'ExpressionAttributeValues': {':qty': {'N': '2'}}
            }
        },
        {
            'Put': {
                'TableName': 'Orders',
                'Item': {
                    'customerId': {'S': 'cust-002'},
                    'orderId':    {'S': 'order-002'},
                    'status':     {'S': 'CONFIRMED'},
                    'amount':     {'N': '29.99'}
                }
            }
        }
    ]
)
print("Transaction committed — inventory decremented and order created atomically")

In [ ]:
# Enable TTL on a sessions table
client.update_time_to_live(
    TableName='Sessions',
    TimeToLiveSpecification={
        'Enabled': True,
        'AttributeName': 'expiresAt'  # Unix timestamp in seconds
    }
)

# Store a session that expires in 30 minutes
import time
sessions_table = dynamodb.Table('Sessions')
sessions_table.put_item(Item={
    'sessionId': 'sess-abc123',
    'userId':    'cust-001',
    'expiresAt': int(time.time()) + 1800  # now + 30 minutes
})
print("Session stored with 30-minute TTL")

## Summary

| Concept | Key Takeaway |
|---|---|
| Primary key — simple | Partition key only; must be unique per item |
| Primary key — composite | PK + SK; multiple items per partition key, unique PK+SK combo |
| Hot partition | Low-cardinality partition key; all traffic hits one partition → throttling |
| Provisioned mode | Set RCU/WCU; cheaper for steady traffic; can throttle |
| On-demand mode | Pay per request; instant scaling; no throttling; costlier per request |
| 1 RCU | 1 strongly consistent read/sec for 4 KB; or 2 eventually consistent |
| 1 WCU | 1 write/sec for 1 KB |
| LSI | Same PK, different SK; created at table creation; supports strong consistency |
| GSI | Different PK; add anytime; eventually consistent only; own throughput |
| Query | By partition key; efficient; always prefer over Scan |
| Scan | Full table read; expensive; avoid in production |
| Transactions | ACID across 100 items/tables; 2× cost |
| DAX | In-memory DynamoDB cache; microsecond reads; write-through; drop-in SDK |
| DynamoDB Streams | Change capture; 24h retention; trigger Lambda; basis for Global Tables |
| Global Tables | Multi-region, multi-active; last-writer-wins; requires Streams |
| TTL | Auto-delete expired items; free; up to 48h delay; appears in Streams |
| PITR | 35-day continuous backup; restore to any second; creates new table |